# CART from scratch

Gini impurity and binary splits for numeric or categorical features. Numeric thresholds are tried between sorted values.

**No scikit-learn is used.** Requirements: `pandas`, `openpyxl`, `numpy`, `matplotlib`. Put `data.xlsx` beside the notebook. Change `TARGET` if the last column is not your class label.

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from math import log2

FILE = "data.xlsx"
df = pd.read_excel(FILE).dropna().copy()
print(df.head())
print("Columns:", list(df.columns))

TARGET = df.columns[-1]  # change if needed
MAX_DEPTH = 4

def gini(y):
    n = len(y)
    return 1 - sum((count/n)**2 for count in Counter(y).values()) if n else 0

def majority(y):
    return Counter(y).most_common(1)[0][0]

def best_split(data, features, target):
    best = None
    parent_gini = gini(data[target])
    for col in features:
        vals = list(data[col].dropna().unique())
        # Numeric columns: try midpoints between sorted unique values.
        if pd.api.types.is_numeric_dtype(data[col]):
            vals = sorted(vals)
            candidates = [(vals[i] + vals[i+1])/2 for i in range(len(vals)-1)]
            for t in candidates:
                left = data[data[col] <= t]
                right = data[data[col] > t]
                if len(left) == 0 or len(right) == 0: continue
                score = len(left)/len(data)*gini(left[target]) + len(right)/len(data)*gini(right[target])
                if best is None or score < best[0]:
                    best = (score, col, t, left, right, "numeric")
        else:
            # Binary categorical split: one category vs all other categories.
            for v in vals:
                left = data[data[col] == v]
                right = data[data[col] != v]
                if len(left) == 0 or len(right) == 0: continue
                score = len(left)/len(data)*gini(left[target]) + len(right)/len(data)*gini(right[target])
                if best is None or score < best[0]:
                    best = (score, col, v, left, right, "categorical")
    if best is None or best[0] >= parent_gini:
        return None
    return best

def cart_tree(data, features, target, depth=0):
    y = data[target]
    if y.nunique() == 1 or depth >= MAX_DEPTH or not features:
        return majority(y)
    split = best_split(data, features, target)
    if split is None:
        return majority(y)
    _, col, value, left, right, kind = split
    return {"feature": col, "value": value, "kind": kind,
            "left": cart_tree(left, features, target, depth+1),
            "right": cart_tree(right, features, target, depth+1),
            "fallback": majority(y)}

def predict_one(tree, row):
    if not isinstance(tree, dict): return tree
    val = row[tree["feature"]]
    if tree["kind"] == "numeric":
        go_left = val <= tree["value"]
    else:
        go_left = val == tree["value"]
    return predict_one(tree["left"] if go_left else tree["right"], row)

features = [c for c in df.columns if c != TARGET]
tree = cart_tree(df, features, TARGET)
pred = df.apply(lambda r: predict_one(tree, r), axis=1)
print("CART tree:", tree)
print("Training accuracy:", round((pred == df[TARGET]).mean(), 3))
pd.Series(pred).value_counts().plot(kind="bar", title="CART predicted class counts")
plt.xlabel("Predicted class"); plt.ylabel("Count"); plt.tight_layout(); plt.show()
